# ¿Qué hace que una vivienda sea más cara?
### Análisis y predicción del precio de la vivienda en México

**Fuente de datos:** API del SNIIV (SEDATU) — `https://sniiv.sedatu.gob.mx/api/CuboAPI/`

Este cuaderno reproduce, paso a paso y de forma manual, los seis requisitos del proyecto:

1. Obtener los datos
2. Limpieza con Python + Pandas
3. Estadística descriptiva
4. Correlaciones
5. Regresión lineal
6. Conclusiones

> Ejecuta las celdas en orden con `Shift + Enter`.


## 0. Librerías

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("Librerías cargadas correctamente")


## Requisito 1 — Obtener los datos

Descargamos cuatro endpoints de la API del SNIIV: Financiamientos, INFONAVIT, CNBV y FOVISSSTE (cobertura nacional, 2024–2025).


In [ ]:
base_url = "https://sniiv.sedatu.gob.mx/api/CuboAPI"

endpoints = {
    'financiamientos': "GetFinanciamiento/2024,2025/00/000/valor_vivienda,modalidad,genero,organismo,destino_credito",
    'infonavit': "GetInfonavit/2024,2025/00/000/valor_vivienda,modalidad,genero,vivienda,linea_credito,rango_salarial,rango_edad",
    'cnbv': "GetCNBV/2024,2025/00/000/valor_vivienda,modalidad,genero,tipo_credito,rango_salarial",
    'fovissste': "GetFOVISSSTE/2024,2025/00/000/valor_vivienda,modalidad,genero,esquema,rango_salarial",
}

frames = []
for nombre, ep in endpoints.items():
    r = requests.get(f"{base_url}/{ep}", timeout=60)
    df = pd.DataFrame(r.json())
    df['fuente'] = nombre
    frames.append(df)
    print(f"{nombre:<16}: {len(df):>6} registros")

df_crudo = pd.concat(frames, ignore_index=True)
print(f"{'TOTAL':<16}: {len(df_crudo):>6} registros")
df_crudo.head()


## Requisito 2 — Limpieza con Python + Pandas

Renombramos columnas, tratamos "No disponible" y nulos, y calculamos el monto unitario (proxy del precio) como `monto / acciones`.


In [ ]:
df = df_crudo.rename(columns={'año': 'anio', 'sexo': 'genero', 'valor_vivienda': 'segmento',
                              'monto': 'monto_financiamiento', 'acciones': 'num_acciones'})

# Tratar valores "No disponible" y vacíos
df.replace('No disponible', np.nan, inplace=True)
df.replace('', np.nan, inplace=True)

# Convertir a numérico
df['monto_financiamiento'] = pd.to_numeric(df['monto_financiamiento'], errors='coerce')
df['num_acciones'] = pd.to_numeric(df['num_acciones'], errors='coerce')

# Monto unitario = precio promedio por financiamiento
df['monto_unitario'] = df['monto_financiamiento'] / df['num_acciones']
df['monto_unitario'] = df['monto_unitario'].replace([np.inf, -np.inf], np.nan)

# Filtrado de registros válidos
df_clean = df.dropna(subset=['monto_unitario', 'segmento', 'modalidad', 'genero']).copy()
df_clean = df_clean[df_clean['monto_unitario'] > 0]
df_clean = df_clean[df_clean['num_acciones'] > 0]

print(f"Registros crudos : {len(df):>7}")
print(f"Registros limpios: {len(df_clean):>7}")
df_clean.head()


### Estadística descriptiva global

In [ ]:
precio = df_clean['monto_unitario']

print(f"N                : {precio.count():>12,}")
print(f"Media            : ${precio.mean():>12,.0f}")
print(f"Mediana          : ${precio.median():>12,.0f}")
print(f"Desv. estándar   : ${precio.std():>12,.0f}")
print(f"Mínimo           : ${precio.min():>12,.0f}")
print(f"Q1 (25%)         : ${precio.quantile(0.25):>12,.0f}")
print(f"Q3 (75%)         : ${precio.quantile(0.75):>12,.0f}")
print(f"Máximo           : ${precio.max():>12,.0f}")
print(f"Asimetría (skew) : {precio.skew():>12.3f}")
print(f"Curtosis         : {precio.kurt():>12.3f}")


### Estadística descriptiva por segmento

In [ ]:
desc_segmento = df_clean.groupby('segmento')['monto_unitario'].agg(
    n='count', media='mean', mediana='median', desv_std='std',
    minimo='min', maximo='max'
).sort_values('media', ascending=False)

desc_segmento


## Requisito 4 — Correlaciones

Codificamos las variables categóricas con *one-hot* y medimos la correlación de cada categoría con el monto unitario.


In [ ]:
cat_cols = ['segmento', 'modalidad', 'genero', 'destino_credito', 'vivienda',
            'linea_credito', 'rango_salarial', 'esquema']
cat_cols = [c for c in cat_cols if c in df_clean.columns]

dummies = pd.get_dummies(df_clean[cat_cols].fillna('No disponible').astype(str), prefix_sep='=')
dummies['monto_unitario'] = df_clean['monto_unitario'].values

corr_precio = (dummies.corr()['monto_unitario']
               .drop('monto_unitario')
               .sort_values(key=np.abs, ascending=False))
print("Correlación de cada categoría con el precio (top 12):")
corr_precio.head(12).round(3)


In [ ]:
plt.figure(figsize=(15, 12))
sns.heatmap(dummies.corr(), cmap='coolwarm', center=0, cbar_kws={'shrink': .7})
plt.title('Matriz de correlaciones (variables codificadas)', fontweight='bold')
plt.tight_layout()
plt.show()


## Requisito 5 — Regresión lineal

Modelo de regresión lineal sobre las variables codificadas, con métricas y análisis de residuos.


In [ ]:
X = dummies.drop(columns='monto_unitario').values
y = dummies['monto_unitario'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo = make_pipeline(StandardScaler(), LinearRegression())
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"R²   : {r2:.4f}  (explica el {r2*100:.1f}% de la variabilidad)")
print(f"RMSE : ${rmse:,.0f}")
print(f"MAE  : ${mae:,.0f}")


In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, s=12, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Predicción perfecta')
plt.xlabel('Monto real (MXN)')
plt.ylabel('Monto predicho (MXN)')
plt.title(f'Regresión Lineal: Predicho vs Real (R²={r2:.3f})', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
coefs = pd.DataFrame({
    'variable': dummies.drop(columns='monto_unitario').columns,
    'coeficiente': modelo.named_steps['linearregression'].coef_
}).sort_values('coeficiente', key=np.abs, ascending=False)

print("Coeficientes más influyentes:")
coefs.head(12).round(0)


## Modelos complementarios y validación cruzada

Comparamos la regresión lineal con Random Forest y Gradient Boosting, y validamos con 5-fold.


In [ ]:
modelos = {
    'Regresión Lineal': modelo,
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
}

resultados = []
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    p = m.predict(X_test)
    resultados.append({
        'Modelo': nombre,
        'RMSE': np.sqrt(mean_squared_error(y_test, p)),
        'MAE': mean_absolute_error(y_test, p),
        'R2': r2_score(y_test, p),
    })

pd.DataFrame(resultados).set_index('Modelo').round(4)


In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf, X, y, cv=cv, scoring='r2')
print(f"Validación cruzada (5-fold) - Random Forest")
print(f"R² medio: {scores.mean():.4f} ± {scores.std():.4f}")


## Requisito 6 — Conclusiones


In [ ]:
print("CONCLUSIONES")
print("=" * 60)
print(f"1. Distribución con asimetría positiva (skew={precio.skew():.2f}):")
print(f"   media (${precio.mean():,.0f}) > mediana (${precio.median():,.0f}).")
top_corr = corr_precio.index[0]
print(f"2. Mayor correlación con el precio: {top_corr} (r={corr_precio.iloc[0]:.2f}).")
top_seg = desc_segmento.index[0]
print(f"3. Segmento más caro: {top_seg} (${desc_segmento.loc[top_seg, 'media']:,.0f}).")
print(f"   Segmento más barato: {desc_segmento.index[-1]} (${desc_segmento['media'].min():,.0f}).")
print(f"4. Regresión lineal R²={r2:.3f}; Random Forest R²={max(r['R2'] for r in resultados):.3f}.")
print()
print("El precio de la vivienda se explica principalmente por el SEGMENTO")
print("(económica < popular < tradicional < media < residencial < residencial plus),")
print("seguido de la modalidad (nueva/existente) y el nivel salarial del acreditado.")
